# Lakehouse and catalogs

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/concepts/lakehouse-catalogs.ipynb)

Separate object storage, table metadata, catalog lookup and authorization. A table location tells an engine where files live; a catalog resolves a name to the metadata needed to read and commit.

## Decisions and trade-offs

Unity Catalog, Polaris, Glue and REST catalog implementations have different authentication and governance models. A REST protocol is an interface, not a guarantee that two services expose identical permissions or features. Check engine support, optimistic concurrency, credential vending, audit trails and recovery procedures.

Plan migration by recording old names, locations, ownership and required grants. Test reads and writes from each engine before changing production clients. Never treat an object-store path alone as authorization.

[Iceberg REST catalog specification](https://iceberg.apache.org/rest-catalog-spec/)

## Worked example

Run the small fixture and inspect each assertion. Change one input to see which contract fails.

In [ ]:
# Optimistic metadata-pointer model, not a running catalog service.
catalog={'orders':{'metadata':'snapshot-1','revision':1}}
def commit(name, expected, metadata):
    current=catalog[name]
    if current['revision']!=expected:
        raise ValueError('stale writer; reload and retry')
    catalog[name]={'metadata':metadata,'revision':expected+1}
commit('orders',1,'snapshot-2')
try: commit('orders',1,'snapshot-3')
except ValueError: pass
else: raise AssertionError('lost-update protection failed')
assert catalog['orders']['metadata']=='snapshot-2' 

## Practice

Add a failure case, explain the expected outcome before running it, and identify which guarantee would need a real service or a larger dataset to verify.